# Ensemble Learning für DSS-Daten

In diesem Notebook konzentrieren wir uns auf die Modellierung der Schweregrade mittels Ensemble-Methoden. Wir nutzen eine Kombination aus Random Forest, Support Vector Machine und Logistic Regression über einen Voting Classifier, um die Robustheit der Vorhersage zu maximieren.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

sns.set_style('whitegrid')

## 1. Daten laden und Labels generieren

Wir laden die Daten und nutzen K-Means, um die drei Belastungskategorien (Gesund, Moderat, Schwer) zu identifizieren.

In [ ]:
# Daten laden
df = pd.read_csv('testdata.txt', sep='\t')

# Features extrahieren
X = df[['bwc', 'vwr']].values

# K-Means für Labeling (wie in der vorherigen Analyse)
scaler_labels = StandardScaler()
X_scaled_all = scaler_labels.fit_transform(X)

kmeans = KMeans(n_clusters=3, random_state=42, n_init=20)
cluster_labels = kmeans.fit_predict(X_scaled_all)
centers = scaler_labels.inverse_transform(kmeans.cluster_centers_)

# Labels map (0: Gesund, 1: Moderat, 2: Schwer) basierend auf BWC und VWR score
score = centers[:, 0] + centers[:, 1]
order = np.argsort(score)
mapping = {order[0]: 2, order[1]: 1, order[2]: 0}
df['severity'] = np.array([mapping[c] for c in cluster_labels])

print("Verteilung der Klassen:", df['severity'].value_counts().sort_index().to_dict())

## 2. Train/Test-Split und Skalierung
Hier trennen wir die Daten strikt, um Informationsabfluss (Data Leakage) während des Trainings unseres Ensemble-Modells zu vermeiden.

In [ ]:
y = df['severity'].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler_ml = StandardScaler()
X_train_scaled = scaler_ml.fit_transform(X_train)
X_test_scaled = scaler_ml.transform(X_test)

## 3. Ensemble Modellierung (Random Forest & Voting Classifier)
Wir definieren einzelne Modelle und führen sie in einem **Voting Classifier** zusammen. Dieser wählt die finale Klasse basierend auf der Mehrheitsentscheidung (Soft Voting) der Untermodelle.

In [ ]:
# Basis-Modelle
rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
lr = LogisticRegression(max_iter=1000, random_state=42)
svm = SVC(kernel='linear', probability=True, random_state=42)

# Ensemble: Voting Classifier
ensemble_model = VotingClassifier(
    estimators=[('Random Forest', rf), ('Logistic Regression', lr), ('SVM', svm)],
    voting='soft'  # soft voting nutzt die vorhergesagten Wahrscheinlichkeiten für die Entscheidung
)

# Training
ensemble_model.fit(X_train_scaled, y_train)
print("Modelle erfolgreich trainiert!")

## 4. Evaluation des Ensembles

In [ ]:
y_pred = ensemble_model.predict(X_test_scaled)

print(f"Ensemble Accuracy: {accuracy_score(y_test, y_pred):.3f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Gesund', 'Moderat', 'Schwer']))

# Confusion Matrix visualisieren
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Gesund', 'Moderat', 'Schwer'],
            yticklabels=['Gesund', 'Moderat', 'Schwer'])
plt.ylabel('Tatsächlich')
plt.xlabel('Vorhergesagt')
plt.title('Confusion Matrix - Ensemble Model')
plt.show()